In [1]:
# TODO: add link to readme

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import rasterio

from exactextract import exact_extract

import sys

# use absolute path here
project_path = "/mnt/School/PhD/AI221/Project/"
sys.path.insert(0, project_path)

from src.data_extraction.utils.constants import data_path

In [3]:
rainfall_folder = "rainfall"
rainfall_path = os.path.join(data_path, "rainfall")

In [4]:
ph_bounds = gpd.read_file(os.path.join(data_path,"ph_adm3_municities/PH_Adm3_MuniCities.shp.shp")).to_crs(epsg=4326)
ph_bounds = ph_bounds[ph_bounds["geo_level"] == "City"]
ph_bounds.head()

,adm1_psgc,adm2_psgc,adm3_psgc,adm3_en,geo_level,len_crs,area_crs,len_km,area_km2,geometry
4,100000000,102800000,102805000,City of Batac,City,66661,158252391,66,158.0,"POLYGON ((120.61242 18.10947, 120.612 18.10679..."
11,100000000,102800000,102812000,City of Laoag,City,53964,110146974,53,110.0,"POLYGON ((120.62081 18.22355, 120.62111 18.223..."
28,100000000,102900000,102906000,City of Candon,City,62247,77652664,62,77.0,"POLYGON ((120.46394 17.23489, 120.46412 17.234..."
56,100000000,102900000,102934000,City of Vigan,City,25067,24485368,25,24.0,"POLYGON ((120.37703 17.58121, 120.37817 17.581..."
70,100000000,103300000,103314000,City of San Fernando,City,54233,99006121,54,99.0,"POLYGON ((120.42381 16.6435, 120.42522 16.6417..."


## Creating CHIRPS-exclusive data

In [5]:
# 3. Prepare the Output Directory
output_dir = "chirps_data"
os.makedirs(
    os.path.join(rainfall_path, output_dir),
    exist_ok=True
)

In [6]:
file_dict = {}

for file in os.listdir(rainfall_path):
    if ".tif" not in file:
        continue

    date_val = "-".join(file.split(".")[2:4])
    file_dict[date_val] = os.path.join(rainfall_path, file)

file_dict = dict(sorted(file_dict.items(), key=lambda item: item[1]))
file_dict

{'2022-01': '../../data/rainfall/chirp-v3.0.2022.01.tif',
 '2022-02': '../../data/rainfall/chirp-v3.0.2022.02.tif',
 '2022-03': '../../data/rainfall/chirp-v3.0.2022.03.tif',
 '2022-04': '../../data/rainfall/chirp-v3.0.2022.04.tif',
 '2022-05': '../../data/rainfall/chirp-v3.0.2022.05.tif',
 '2022-06': '../../data/rainfall/chirp-v3.0.2022.06.tif',
 '2022-07': '../../data/rainfall/chirp-v3.0.2022.07.tif',
 '2022-08': '../../data/rainfall/chirp-v3.0.2022.08.tif',
 '2022-09': '../../data/rainfall/chirp-v3.0.2022.09.tif',
 '2022-10': '../../data/rainfall/chirp-v3.0.2022.10.tif',
 '2022-11': '../../data/rainfall/chirp-v3.0.2022.11.tif',
 '2022-12': '../../data/rainfall/chirp-v3.0.2022.12.tif',
 '2023-01': '../../data/rainfall/chirp-v3.0.2023.01.tif',
 '2023-02': '../../data/rainfall/chirp-v3.0.2023.02.tif',
 '2023-03': '../../data/rainfall/chirp-v3.0.2023.03.tif',
 '2023-04': '../../data/rainfall/chirp-v3.0.2023.04.tif',
 '2023-05': '../../data/rainfall/chirp-v3.0.2023.05.tif',
 '2023-06': '.

In [7]:
out_dir = "chirps_data"        # Output folder (same as before)
os.makedirs(out_dir, exist_ok=True)

In [8]:
# Master list
all_records = []

# Loop 1: Iterate through the months
for year_month, tif in file_dict.items():
    print(f"Processing {year_month}...")

    # 1. Open the raster ONCE per month
    with rasterio.open(tif) as src:
        means = exact_extract(src, ph_bounds, 'mean')
        
    # 3. Zip the results directly with the PSGC column
    for psgc, mean_val in zip(ph_bounds["adm3_psgc"], means):
        all_records.append({
            'adm3_psgc': psgc,
            'date': year_month,
            'avg': mean_val
        })

print("Extraction complete!")

Processing 2022-01...
Processing 2022-02...
Processing 2022-03...
Processing 2022-04...
Processing 2022-05...
Processing 2022-06...
Processing 2022-07...
Processing 2022-08...
Processing 2022-09...
Processing 2022-10...
Processing 2022-11...
Processing 2022-12...
Processing 2023-01...
Processing 2023-02...
Processing 2023-03...
Processing 2023-04...
Processing 2023-05...
Processing 2023-06...
Processing 2023-07...
Processing 2023-08...
Processing 2023-09...
Processing 2023-10...
Processing 2023-11...
Processing 2023-12...
Processing 2024-01...
Processing 2024-02...
Processing 2024-03...
Processing 2024-04...
Processing 2024-05...
Processing 2024-06...
Processing 2024-07...
Processing 2024-08...
Processing 2024-09...
Processing 2024-12...
Processing 2025-01...
Processing 2025-02...
Processing 2025-03...
Processing 2025-04...
Processing 2025-05...
Processing 2025-06...
Processing 2025-07...
Processing 2025-08...
Processing 2025-09...
Processing 2025-10...
Processing 2025-11...
Processing

In [10]:
# 4. Consolidate to one dataframe
print("\nCompiling and saving individual municipality CSVs...")
master_df = pd.DataFrame(all_records)
master_df["avg"] = master_df["avg"].apply(lambda x: x.get("properties").get("mean"))
master_df.head()


Compiling and saving individual municipality CSVs...


,adm3_psgc,date,avg
0,102805000,2022-01,27.651221
1,102812000,2022-01,26.956228
2,102906000,2022-01,26.493692
3,102934000,2022-01,25.126945
4,103314000,2022-01,16.391532


In [13]:
master_df.adm3_psgc.nunique(), master_df.date.nunique()

(149, 48)

In [14]:
# Validation: should have no negative values
master_df[master_df["avg"] < 0]

,adm3_psgc,date,avg


In [15]:
# Validation: some records have months w/o rain
master_df[master_df["avg"] == 0]

,adm3_psgc,date,avg
23,305409000,2022-01,0.0
27,331400000,2022-01,0.0
1493,102934000,2022-11,0.0
1642,102934000,2022-12,0.0
1645,105518000,2022-12,0.0
...,...,...,...
4061,402109000,2024-04,0.0
4144,1380200000,2024-04,0.0
4152,1381000000,2024-04,0.0
6708,102934000,2025-12,0.0


In [16]:
master_df.to_csv(
    os.path.join(data_path, "urban_monthly_rainfall.csv"), 
    index=False
)